# Evaluating LLM-Generated Synthetic Datasets for SMS Smishing Detection

This notebook walks through the full ML pipeline for comparing SMS smishing classifiers trained on:
- **Manual-only**: manually curated real-world SMS data
- **Synthetic-only**: LLM-generated synthetic SMS data (size-matched to manual)
- **Combined**: both datasets together

All experiments are evaluated against a manual (real-world) holdout set across 30 random seeds and 3 classifier types.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable when running from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

## 2. Dataset Loading & Preprocessing

In [ ]:
from src.data.preprocessing import run_preprocessing

manual, synthetic, combined = run_preprocessing(mode="drop_exact_duplicates", save=True)

print(f"Manual:    {len(manual):,} rows")
print(f"Synthetic: {len(synthetic):,} rows")
print(f"Combined:  {len(combined):,} rows")

## 3. Dataset Audit & Provenance Analysis

The synthetic dataset was generated using the manual dataset as source material. We audit for duplicates, overlap, and potential leakage.

In [ ]:
from src.data.audit import run_audit

audit_results = run_audit(manual, synthetic, save=True)

display(Markdown("### Dataset Summary"))
display(audit_results["audit"])

In [ ]:
display(Markdown("### Leakage Audit"))
display(audit_results["leakage"])

In [ ]:
display(Markdown("### Top Duplicate Messages (across both datasets)"))
display(audit_results["top_duplicates"].head(10))

## 4. Duplicate Analysis

In [ ]:
for name, df in [("Manual", manual), ("Synthetic", synthetic)]:
    n_total = len(df)
    n_unique = df["text"].nunique()
    print(f"{name}: {n_total:,} rows, {n_unique:,} unique texts, {n_total - n_unique:,} duplicates removed")

## 5. Cross-Dataset Overlap

In [ ]:
overlap = audit_results["overlap"]
print(f"Exact text matches between datasets: {len(overlap):,}")
print(f"Overlap as % of manual: {len(overlap) / manual['text'].nunique() * 100:.1f}%")
print(f"Overlap as % of synthetic: {len(overlap) / synthetic['text'].nunique() * 100:.1f}%")

## 6. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (name, df) in zip(axes, [("Manual", manual), ("Synthetic", synthetic)]):
    counts = df["label"].value_counts().reindex(["ham", "spam", "smishing"])
    counts.plot.bar(ax=ax, color=["steelblue", "orange", "firebrick"])
    ax.set_title(f"{name} Dataset")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 7. Feature Engineering

Each model uses a scikit-learn pipeline that combines:
- **TF-IDF** vectorization on `text_clean` (unigrams + bigrams, sublinear TF)
- **Indicator features**: `has_url`, `has_email`, `has_phone`

In [ ]:
from src.features.text_features import build_feature_transformer

ft = build_feature_transformer()
print("Feature transformer:")
print(ft)

## 8. Model Training

We train 3 classifiers across 3 training strategies and 30 random seeds (270 total runs).

**Models**: MultinomialNB, Logistic Regression, LinearSVC (calibrated)

**Training strategies** (all tested against Manual Holdout):
- `manual_only` — trained on manual data only
- `synthetic_only` — trained on synthetic data (size-matched to manual)
- `combined` — trained on manual + full synthetic data

In [ ]:
from src.models.train import run_experiments

metrics_df = run_experiments(manual, synthetic, verbose=True)

## 9. Model Evaluation

In [ ]:
display(Markdown("### Mean Smishing F1 by Model and Training Strategy"))
summary = (
    metrics_df.groupby(["model_name", "experiment_id"])["smishing_f1"]
    .agg(["mean", "std"])
    .round(4)
    .unstack()
)
display(summary)

In [ ]:
display(Markdown("### Mean Macro F1 by Model and Training Strategy"))
summary_macro = (
    metrics_df.groupby(["model_name", "experiment_id"])["macro_f1"]
    .agg(["mean", "std"])
    .round(4)
    .unstack()
)
display(summary_macro)

In [ ]:
display(Markdown("### Training Set Size Verification"))
size_check = (
    metrics_df.groupby("experiment_id")[["n_train", "n_test"]]
    .agg(["mean", "min", "max"])
    .round(0)
)
display(size_check)

## 10. Evaluation Figures

In [ ]:
from src.models.figures import generate_all_figures

generate_all_figures(metrics_df, manual, synthetic)
print("All evaluation figures saved to outputs/figures/")

In [ ]:
from IPython.display import Image

display(Markdown("### Smishing F1 by Training Strategy"))
display(Image(filename=str(project_root / "outputs/figures/mean_smishing_f1_by_training_strategy.png")))

In [ ]:
display(Markdown("### Smishing F1 Boxplot"))
display(Image(filename=str(project_root / "outputs/figures/smishing_f1_boxplot_by_training_strategy.png")))

In [ ]:
display(Markdown("### Best Model Confusion Matrix (Manual Holdout)"))
display(Image(filename=str(project_root / "outputs/figures/confusion_matrix_best_model_manual_holdout.png")))

## 11. Statistical Summary

Detailed statistical hypothesis testing is performed in R (`r/statistical_analysis.R`). Here we preview the key comparisons.

In [ ]:
from scipy import stats

display(Markdown("### Quick Paired Comparisons (smishing F1, paired by seed+model)"))

comparisons = [
    ("manual_only", "synthetic_only"),
    ("manual_only", "combined"),
    ("synthetic_only", "combined"),
]

for exp_a, exp_b in comparisons:
    a = metrics_df[metrics_df["experiment_id"] == exp_a].sort_values(["seed", "model_name"])["smishing_f1"].values
    b = metrics_df[metrics_df["experiment_id"] == exp_b].sort_values(["seed", "model_name"])["smishing_f1"].values
    t_stat, p_val = stats.ttest_rel(a, b)
    diff = b.mean() - a.mean()
    print(f"{exp_a} vs {exp_b}: mean diff = {diff:+.4f}, t = {t_stat:.3f}, p = {p_val:.2e}")

## 12. Best Model Selection

In [ ]:
import json

with open(project_root / "outputs/models/best_model_metadata.json") as f:
    meta = json.load(f)

display(Markdown("### Best Model Metadata"))
for k, v in meta.items():
    print(f"  {k}: {v}")

## 13. Artifact Export Summary

All outputs have been saved:

| Category | Files |
|----------|-------|
| Processed data | `data/processed/manual_clean.csv`, `synthetic_clean.csv`, `combined_clean.csv` |
| Metrics | `outputs/metrics/all_model_results.csv`, `confusion_matrices.csv`, `classification_reports.jsonl` |
| Tables | `outputs/tables/dataset_audit.csv`, `leakage_audit.csv`, `model_performance_summary.csv`, etc. |
| Figures | `outputs/figures/*.png` (8 figures) |
| Model | `outputs/models/best_model.joblib`, `best_model_metadata.json` |

In [ ]:
import os

for dirpath in ["data/processed", "outputs/metrics", "outputs/tables", "outputs/figures", "outputs/models"]:
    full = project_root / dirpath
    files = sorted(os.listdir(full)) if full.exists() else []
    print(f"\n{dirpath}/")
    for f in files:
        size = (full / f).stat().st_size
        print(f"  {f} ({size:,} bytes)")